# importy i funckje

In [21]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import pandas_ta as ta
from tqdm import tqdm
import numpy as np

# widgety do wyboru ustawień (interwal, filtry)
from maciex_py.poczatek_ustawienia import create_settings_ui

# kontrola taliba
from maciex_py.kontrola_taliba import create_talib_control

# zaladowanie df do pamieci
from maciex_py.ladowanie_danych import create_stock_dfs, add_indicators

# dodanie formacji
from maciex_py.dodanie_formacji import add_candle_patterns

# giga plot
from maciex_py.main_plot import create_chart_ui


In [22]:
# Indicator functions from pandas-ta
def hma(close, length):
    return ta.hma(close, length=length)

def rsi(close, length):
    return ta.rsi(close, length=length)

def aroon(high, low, length):
    result = ta.aroon(high, low, length=length)
    # Return just the Aroon Up indicator
    aroon_cols = [col for col in result.columns if 'AROONU' in col]
    return result[aroon_cols[0]] if aroon_cols else pd.Series(0, index=high.index)

def williams_r(high, low, close, length):
    return ta.willr(high, low, close, length=length)

def supertrend(df):
    result = ta.supertrend(df['High'], df['Low'], df['Close'], length=10, multiplier=3)
    supertrend_cols = [col for col in result.columns if 'SUPERT' in col]
    return result[supertrend_cols[0]] if supertrend_cols else pd.Series(0, index=df.index)


# poczatkowe ustawienia

In [23]:
settings_panel, settings = create_settings_ui()
display(settings_panel)

In [24]:
print("Wybrane ustawienia:")
for key, value in settings.items():
    print(f"{key}: {value}")

Wybrane ustawienia:
market: stocks
interval: 1d
week_start: WED
vol_enabled: True
vol_ratio_window: 20
vol_ratio_threshold: 1.2000000000000002
cmo_enabled: True
cmo_len: 6
cmo_thres: -35
cmo_thres_prev: -50


# kontrola taliba

In [25]:
talib_panel, apply_params_fn, registry = create_talib_control(settings)
display(talib_panel)
apply_params_fn()


# zaladowanie df'ow

In [26]:
dfs_1d, dfs_1w = create_stock_dfs(settings)

Loading data:   0%|          | 0/479 [00:00<?, ?it/s]

In [27]:
ex_1d = dfs_1d.get("ALLE")
ex_1w = dfs_1w.get("ISRG")

# plot

In [28]:
settings

{'market': 'stocks',
 'interval': '1d',
 'week_start': 'WED',
 'vol_enabled': True,
 'vol_ratio_window': 20,
 'vol_ratio_threshold': 1.2000000000000002,
 'cmo_enabled': True,
 'cmo_len': 6,
 'cmo_thres': -35,
 'cmo_thres_prev': -50}

In [29]:
symbols = list(dfs_1d.keys())
ui = create_chart_ui(dfs_1d, dfs_1w, settings, symbols, 
                      add_indicators, add_candle_patterns,
                      hma, rsi, aroon, williams_r, supertrend)
display(ui)

# hehe statystyka

## ladny print

In [30]:
trades_df

NameError: name 'trades_df' is not defined

In [ ]:
def print_trade_results_stats(trades_df):
    if trades_df is None or trades_df.empty:
        print("❌ No trades to analyze.")
        return

    df = trades_df.copy()

    total_trades = len(df)
    wins = (df["return_pct"] > 0).sum()
    losses = (df["return_pct"] <= 0).sum()
    win_rate = wins / total_trades * 100 if total_trades else 0

    print("=" * 80)
    print("FIRST PROFIT STRATEGY — OVERALL PERFORMANCE")
    print("=" * 80)

    print(f"Trades        : {total_trades}")
    print(f"Wins / Losses : {wins} / {losses}")
    print(f"Win rate      : {win_rate:.2f}%")
    print(f"Avg return    : {df['return_pct'].mean():.2f}%")
    print(f"Median return : {df['return_pct'].median():.2f}%")

    if "hold_bars" in df:
        print(f"Avg hold      : {df['hold_bars'].mean():.2f} bars")

    # ======================================================================
    # EXIT REASONS
    # ======================================================================

    if "exit_reason" in df:
        print("\n" + "=" * 80)
        print("EXIT REASONS")
        print("=" * 80)

        exit_stats = df.groupby("exit_reason").agg(
            trades=("return_pct", "count"),
            win_rate=("return_pct", lambda x: (x > 0).mean() * 100),
            avg_return_pct=("return_pct", "mean"),
        )

        exit_stats["occurrence_pct"] = (
            exit_stats["trades"] / total_trades * 100
        )

        print(exit_stats.sort_values("trades", ascending=False).round(2))

    # ======================================================================
    # PATTERN PERFORMANCE
    # ======================================================================

    if "pattern" in df:
        print("\n" + "=" * 80)
        print("PATTERN PERFORMANCE")
        print("=" * 80)

        pattern_stats = df.groupby("pattern").agg(
            trades=("return_pct", "count"),
            win_rate=("return_pct", lambda x: (x > 0).mean() * 100),
            avg_return_pct=("return_pct", "mean"),
        )

        pattern_stats["occurrence_pct"] = (
            pattern_stats["trades"] / total_trades * 100
        )

        print(
            pattern_stats
            .sort_values("avg_return_pct", ascending=False)
            .round(2)
        )

    # ======================================================================
    # DURATION METRICS
    # ======================================================================

    if "hold_bars" in df:
        print("\n" + "=" * 80)
        print("TRADE DURATION")
        print("=" * 80)

        print(f"Avg hold (bars)    : {df['hold_bars'].mean():.2f}")
        print(f"Median hold (bars) : {df['hold_bars'].median():.0f}")
        print(f"Max hold (bars)    : {df['hold_bars'].max()}")

        if "hold_time" in df and df["hold_time"].notna().any():
            print(f"Avg hold (time)    : {df['hold_time'].mean()}")

        print("\nHold bars by exit reason:")
        print(
            df.groupby("exit_reason")["hold_bars"]
            .agg(["mean", "median", "max"])
            .round(2)
        )

    # ======================================================================
    # RISK METRICS
    # ======================================================================

    print("\n" + "=" * 80)
    print("RISK METRICS")
    print("=" * 80)

    avg_win = df.loc[df["return_pct"] > 0, "return_pct"].mean()
    avg_loss = df.loc[df["return_pct"] <= 0, "return_pct"].mean()

    profit_factor = (
        df.loc[df["return_pct"] > 0, "return_pct"].sum()
        / abs(df.loc[df["return_pct"] <= 0, "return_pct"].sum())
        if losses > 0 else float("inf")
    )

    expectancy = (
        (wins / total_trades) * avg_win +
        (losses / total_trades) * avg_loss
    )

    print(f"Avg win        : {avg_win:.2f}%")
    print(f"Avg loss       : {avg_loss:.2f}%")
    print(f"Profit factor  : {profit_factor:.2f}")
    print(f"Expectancy     : {expectancy:.2f}%")


## patterns_df

In [ ]:
def extract_patterns_by_cols(dfs_dict, cols, desc="Scanning for all patterns"):
    rows = []

    for symbol, df in tqdm(dfs_dict.items(), desc=desc):
        if df is None or df.empty:
            continue
        
        df = add_candle_patterns(df, settings)

        for col in cols:
            if col not in df.columns:
                continue

            series = pd.to_numeric(df[col], errors="coerce")
            hits = series[series > 0]

            for idx, val in hits.items():
                rows.append(
                    {
                        "symbol": symbol,
                        "column": col,
                        "timestamp": idx,
                        "value": val,
                    }
                )

    return pd.DataFrame(rows)

In [ ]:
pattern_cols = ['hammer', 'inverted_hammer', 'engulfing_bull', 'piercing_line']

In [ ]:
patterns_df_1d = extract_patterns_by_cols(
    dfs_1d,
    pattern_cols,
    desc="🔍 Scanning 1D patterns",
)

patterns_df_1w = extract_patterns_by_cols(
    dfs_1w,
    pattern_cols,
    desc="🔍 Scanning 1W patterns",
)


🔍 Scanning 1W patterns: 100%|██████████| 479/479 [00:10<00:00, 46.35it/s]


In [ ]:
patterns_df_1d

,symbol,column,timestamp,value
0,MMM,hammer,2021-04-19,100.0
1,MMM,hammer,2021-05-19,100.0
2,MMM,hammer,2021-05-26,100.0
3,MMM,hammer,2021-10-18,100.0
4,MMM,hammer,2021-10-21,100.0
...,...,...,...,...
45024,ZTS,engulfing_bull,2025-12-30,1.0
45025,ZTS,engulfing_bull,2026-01-05,1.0
45026,ZTS,engulfing_bull,2026-01-30,1.0
45027,ZTS,piercing_line,2022-10-21,100.0


## first profit paczymy

In [ ]:
def first_profit_strat(
    max_hold=10,

    hard_tp=0.02,
    hard_sl=0.08,
    vol_tp_mult=1.0,
    vol_sl_mult=3.0,
    min_profit_exit=0.005,

    use_hard_tp=True,
    use_hard_sl=True,
    use_vol_tp=False,
    use_vol_sl=False,

    require_vol_confirmation=True,
    require_cmo_confirmation=True,

    interval="1d",
    weekly_exit_on_daily=True,
    entry_offset=1,

    pattern_cols=None,
    debug=True,
):
    global patterns_df_1d, patterns_df_1w, dfs_1d, dfs_1w

    entry_offset = max(entry_offset, 1)  # prevent same-candle entry

    patterns = patterns_df_1d if interval == "1d" else patterns_df_1w
    entry_dfs = dfs_1d if interval == "1d" else dfs_1w
    exit_dfs = dfs_1d if (interval == "1w" and weekly_exit_on_daily) else entry_dfs

    patterns = patterns.sort_values("timestamp")

    trades = []

    debug_counts = {
        "no_pattern": 0,
        "symbol_missing": 0,
        "timestamp_missing": 0,
        "vol_reject": 0,
        "cmo_reject": 0,
        "entry_oob": 0,
        "exit_df_missing": 0,
        "future_empty": 0,
    }

    if pattern_cols is None:
        pattern_cols = patterns["column"].unique().tolist()

    for _, row in tqdm(
        patterns.iterrows(),
        total=len(patterns),
        desc="📊 Building trades",
    ):
        symbol = row["symbol"]
        ts = row["timestamp"]
        pattern = row["column"]

        # ===== PATTERN FILTER =====
        if pattern not in pattern_cols:
            debug_counts["no_pattern"] += 1
            continue

        # ===== SYMBOL =====
        if symbol not in entry_dfs:
            debug_counts["symbol_missing"] += 1
            continue

        entry_df = entry_dfs[symbol]
        if ts not in entry_df.index:
            debug_counts["timestamp_missing"] += 1
            continue

        sig = entry_df.loc[ts]

        # ===== CONFIRMATIONS =====
        if require_vol_confirmation:
            v = sig.get("VOL_SIGNIFICANT", 0)
            if isinstance(v, pd.Series):
                v = v.iloc[0]
            if v != 1:
                debug_counts["vol_reject"] += 1
                continue

        if require_cmo_confirmation:
            c = sig.get("DOWNTREND_SHORT", 0)
            if isinstance(c, pd.Series):
                c = c.iloc[0]
            if c != 1:
                debug_counts["cmo_reject"] += 1
                continue

        # ===== ENTRY =====
        entry_loc = entry_df.index.get_loc(ts) + entry_offset
        if entry_loc >= len(entry_df):
            debug_counts["entry_oob"] += 1
            continue

        entry_time = entry_df.index[entry_loc]
        entry_row = entry_df.iloc[entry_loc]
        entry_price = entry_row["Open"]
        atr = entry_row.get("ATR", None)

        # ===== EXIT DATA =====
        exit_df = exit_dfs.get(symbol)
        if exit_df is None:
            debug_counts["exit_df_missing"] += 1
            continue

        future = exit_df.loc[exit_df.index > entry_time].iloc[:max_hold]
        if future.empty:
            debug_counts["future_empty"] += 1
            continue

        # ===== TP / SL =====
        tp = sl = None

        if use_hard_tp:
            tp = entry_price * (1 + hard_tp)
        if use_hard_sl:
            sl = entry_price * (1 - hard_sl)

        if atr is not None:
            if use_vol_tp:
                tp = entry_price + atr * vol_tp_mult
            if use_vol_sl:
                sl = entry_price - atr * vol_sl_mult

        exit_price = future.iloc[-1]["Close"]
        exit_time = future.index[-1]
        exit_reason = "TIME_EXIT"
        exit_loc = future.index.get_loc(exit_time)

        highs = future["High"].values
        lows = future["Low"].values
        closes = future["Close"].values

        # ===== SL FIRST =====
        if sl is not None:
            hit = np.where(lows <= sl)[0]
            if len(hit):
                exit_loc = hit[0]
                exit_price = sl
                exit_time = future.index[exit_loc]
                exit_reason = "SL"

        # ===== TP =====
        if exit_reason == "TIME_EXIT" and tp is not None:
            hit = np.where(highs >= tp)[0]
            if len(hit):
                exit_loc = hit[0]
                exit_price = tp
                exit_time = future.index[exit_loc]
                exit_reason = "TP"

        # ===== MIN PROFIT =====
        if exit_reason == "TIME_EXIT" and min_profit_exit and min_profit_exit > 0:
            profits = (closes - entry_price) / entry_price
            hit = np.where(profits >= min_profit_exit)[0]
            if len(hit):
                exit_loc = hit[0]
                exit_price = closes[exit_loc]
                exit_time = future.index[exit_loc]
                exit_reason = "MIN_PROFIT"

        # ===== METRICS =====
        ret_pct = (exit_price - entry_price) / entry_price * 100
        hold_bars = exit_loc + 1

        hold_time = None
        if isinstance(entry_time, pd.Timestamp) and isinstance(exit_time, pd.Timestamp):
            hold_time = exit_time - entry_time

        trades.append({
            "symbol": symbol,
            "pattern": pattern,
            "entry_time": entry_time,
            "exit_time": exit_time,
            "entry_price": entry_price,
            "exit_price": exit_price,
            "return_pct": ret_pct,
            "exit_reason": exit_reason,
            "hold_bars": hold_bars,
            "hold_time": hold_time,
        })

    trades_df = pd.DataFrame(trades)

    if trades_df.empty and debug:
        print("⚠️ No trades generated. Debug summary:")
        for k, v in debug_counts.items():
            print(f"  {k}: {v}")

    return trades_df


In [ ]:
trades_df = first_profit_strat(
    max_hold=3,

    hard_tp=0.02,
    hard_sl=0.08,
    vol_tp_mult=0.5,
    vol_sl_mult=2.0,
    min_profit_exit=0.005,

    use_hard_tp=True,
    use_hard_sl=True,
    use_vol_tp=True,
    use_vol_sl=True,

    require_vol_confirmation=True,
    require_cmo_confirmation=True,

    interval="1d",
    weekly_exit_on_daily=True,
    entry_offset=0,

    pattern_cols=[
        'hammer',
        'inverted_hammer',
        'engulfing_bull',
        'piercing_line',
        ],  
)


📊 Building trades: 100%|██████████| 45029/45029 [00:07<00:00, 5964.74it/s]


In [ ]:
trades_df

,symbol,pattern,entry_time,exit_time,entry_price,exit_price,return_pct,exit_reason,hold_bars,hold_time
0,FDS,inverted_hammer,2021-04-01,2021-04-05,311.059998,315.333775,1.373940,TP,1,4 days
1,WBD,inverted_hammer,2021-04-15,2021-04-20,38.790001,35.470001,-8.558906,TIME_EXIT,3,5 days
2,STT,hammer,2021-04-20,2021-04-23,80.260002,81.110001,1.059056,MIN_PROFIT,3,3 days
3,LYV,engulfing_bull,2021-04-22,2021-04-26,81.519997,82.927992,1.727178,TP,2,4 days
4,HBAN,piercing_line,2021-04-22,2021-04-23,15.410000,14.340752,-6.938659,SL,1,1 days
...,...,...,...,...,...,...,...,...,...,...
3401,ISRG,hammer,2026-02-05,2026-02-06,482.779999,489.679743,1.429169,TP,1,1 days
3402,PAYC,piercing_line,2026-02-05,2026-02-06,131.149994,133.758836,1.989205,TP,1,1 days
3403,ROK,hammer,2026-02-06,2026-02-09,407.010010,413.445215,1.581093,TP,1,3 days
3404,MOH,inverted_hammer,2026-02-06,2026-02-09,125.970001,131.071110,4.049463,TP,1,3 days


## wyniki

In [ ]:
print_trade_results_stats(trades_df)

FIRST PROFIT STRATEGY — OVERALL PERFORMANCE
Trades        : 3406
Wins / Losses : 2371 / 1035
Win rate      : 69.61%
Avg return    : -0.02%
Median return : 1.08%
Avg hold      : 1.82 bars

EXIT REASONS
             trades  win_rate  avg_return_pct  occurrence_pct
exit_reason                                                  
TP             2201    100.00            1.60           64.62
TIME_EXIT       722      8.03           -2.11           21.20
SL              371      0.00           -5.84           10.89
MIN_PROFIT      112    100.00            0.92            3.29

PATTERN PERFORMANCE
                 trades  win_rate  avg_return_pct  occurrence_pct
pattern                                                          
piercing_line       226     78.76            0.63            6.64
engulfing_bull     1187     69.33            0.12           34.85
hammer             1205     69.21           -0.16           35.38
inverted_hammer     788     68.02           -0.19           23.14

TRADE DUR